In [2]:
import cv2
import numpy as np
import torch
import torchvision.transforms as transforms
import PIL.Image
import time
from torch2trt import TRTModule
from jetbotmini import Robot

import ipywidgets
from IPython.display import display

# 1. 초기화
robot = Robot()
gst_str = ("nvarguscamerasrc ! video/x-raw(memory:NVMM), width=(int)224, height=(int)224, format=(string)NV12, framerate=(fraction)60/1 ! nvvidconv flip-method=0 ! video/x-raw, width=(int)224, height=(int)360, format=(string)BGRx ! videoconvert ! video/x-raw, format=(string)BGR ! appsink")

# 2. TensorRT 전용 모델 불러오기 (TRTModule 사용)
print("TRT 엔진 로딩 중...")
model_trt = TRTModule()
model_trt.load_state_dict(torch.load('best_model_xy_trt.pth'))
device = torch.device('cuda')
model = model_trt.to(device).eval()
print("엔진 로딩 완료!")

mean = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

angle, angle_last = 0.0, 0.0

# 3. 튜닝된 모터 파라미터 (초기 마찰력을 이기는 충분한 힘)
speed_gain_value = 0.33  # 로봇이 너무 빠르거나 느리면 이 값을 0.35 ~ 0.45 사이로 조절하세요.
steering_pgain_value = 0.13  # P gain: 현재 angle 오차에 대한 조향 민감도
steering_dgain_value = 0.05  # D gain: angle 변화량에 대한 흔들림 완화
steering_bias_value = 0.0    # 좌우 기계적 편향 보정

image_widget = ipywidgets.Image(format='jpeg', width=224, height=224)
display(image_widget)

def preprocess(image):
    image = PIL.Image.fromarray(image)
    image = transforms.functional.to_tensor(image).to(device).half()
    image.sub_(mean[:, None, None]).div_(std[:, None, None])
    return image[None, ...]

def executeModel(image):
    global angle, angle_last

    xy = model(preprocess(image)).detach().float().cpu().numpy().flatten()
    x = xy[0]
    y = 0.12

    # y축 기준 목표점 방향각 계산
    angle = np.arctan2(x, y)

    # 본 demo는 I항이 없는 P+D 제어 형태
    p_term = angle * steering_pgain_value
    d_term = (angle - angle_last) * steering_dgain_value
    angle_last = angle

    steering_value = p_term + d_term + steering_bias_value
    
    left_v = max(min(speed_gain_value + steering_value, 1.0), -1.0)
    right_v = max(min(speed_gain_value - steering_value, 1.0), -1.0)

    robot.left_motor.value = left_v
    robot.right_motor.value = right_v

def all_stop():
    robot.stop()

def Video(openpath):
    cap = cv2.VideoCapture(openpath)
    if not cap.isOpened():
        print("카메라 에러: 커널을 재시작해 주세요.")
        return

    print("🚀 TensorRT 자율주행 시작!")
    frame_count = 0

    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            frame_resized = cv2.resize(frame, (224, 224), interpolation=cv2.INTER_AREA)
            
            # TRT 엔진으로 초고속 연산
            executeModel(frame_resized)
            
            # 화면 출력 주기 (15프레임당 1번 출력하여 위젯 과부하 완벽 방지)
            if frame_count % 15 == 0:
                _, jpeg = cv2.imencode('.jpg', frame_resized)
                image_widget.value = jpeg.tobytes()

            frame_count += 1
            time.sleep(0.005)

    except KeyboardInterrupt:
        print("\n정지되었습니다.")
    finally:
        all_stop()
        cap.release()



TRT 엔진 로딩 중...
엔진 로딩 완료!


Image(value=b'', format='jpeg', height='224', width='224')

In [ ]:
Video(gst_str)